# Final Stage F: Ownership, Multiplicity, Counterfactual Replay and 53R2 Correction Audit

**Runtime: CPU. Run all.**

This notebook uses the complete regenerated 2,718-call captures. It compares page-level union, multiplicity and ownership for the seven affected pages, classifies the 50 known coordinates, replays both captured streams, substitutes the nine 51R2 masks into the 53R2 stream, and writes the correction diagnosis plus the regression gates required before Stage F can pass.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy


## 1. Configuration
Edit the two regenerated capture roots. Each root must contain `captured_calls.csv` and `calls/call_XXXXXX/source.npy`, `mask.npy`, and `result.npy`.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, gc
import cv2, numpy as np, pandas as pd
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CAPTURE_51=ROOT/'REPLACE_WITH_REGENERATED_51R2_CAPTURE_ROOT'
CAPTURE_53=ROOT/'REPLACE_WITH_REGENERATED_53R2_CAPTURE_ROOT'
MANIFEST_51=CAPTURE_51/'captured_calls.csv'; MANIFEST_53=CAPTURE_53/'captured_calls.csv'
NINE_MASK_COORDINATES_OVERRIDE=None
COORDINATES=Path(NINE_MASK_COORDINATES_OVERRIDE) if NINE_MASK_COORDINATES_OVERRIDE else ROOT/'stage_f_nine_mask_extract'/'nine_mask_coordinate_differences.csv'
PAGES=['Page003','Page004','Page005','Page008','Page036','Page040','Page049']
EXPECTED={'Page003':2,'Page004':2,'Page005':1,'Page008':1,'Page036':1,'Page040':1,'Page049':1}
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_stage_f_final_audit'
OUT=ROOT/'stage_f_final_ownership_counterfactual'/RUN; T=OUT/'tables'; IMG=OUT/'images'; UP=OUT/'upload'
for p in [OUT,T,IMG,UP]: p.mkdir(parents=True,exist_ok=False)
print('51R2',CAPTURE_51); print('53R2',CAPTURE_53); print('Output',OUT)


## 2. Manifest integrity and nine-call identity


In [ ]:
def req(x,msg):
 if not x: raise RuntimeError(msg)
def ah(a):
 a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def bh(a): return ah((np.asarray(a)>0).astype(np.uint8))
def choose(f,names,label):
 for n in names:
  if n in f.columns:return n
 raise RuntimeError(f'No {label}. Columns: {f.columns.tolist()}')
def page(v):
 s=str(v).strip(); d=''.join(c for c in s if c.isdigit()); return f'Page{int(d):03d}' if d else s
def manifest(path,side):
 req(path.exists(),f'{side}: missing {path}'); f=pd.read_csv(path,low_memory=False)
 c=choose(f,['call_number','call','call_id'],'call number'); p=choose(f,['page','context_page','page_name','page_id'],'page'); m=choose(f,['mask_boolean_sha256','mask_boolean_hash','mask_sha256','mask_hash'],'mask hash')
 o=pd.DataFrame({'call_number':pd.to_numeric(f[c],errors='raise').astype(int),'page':f[p].map(page),'mask_hash':f[m].astype(str)})
 req(len(o)==2718 and o.call_number.nunique()==2718 and o.call_number.min()==1 and o.call_number.max()==2718,f'{side}: not complete unique 1..2718')
 return o.sort_values('call_number').reset_index(drop=True)
def cp(root,c,name): return root/'calls'/f'call_{int(c):06d}'/name
m51=manifest(MANIFEST_51,'51R2'); m53=manifest(MANIFEST_53,'53R2')
j=m51.merge(m53,on='call_number',suffixes=('_51','_53'),validate='one_to_one')
req((j.page_51==j.page_53).all(),'Page identity mismatch')
diff=j[j.mask_hash_51!=j.mask_hash_53].copy(); req(len(diff)==9,f'Expected 9 mask differences, got {len(diff)}')
req(diff.groupby('page_51').size().to_dict()==EXPECTED,'Unexpected differing-page distribution')
diff.to_csv(T/'nine_differing_calls.csv',index=False); display(diff[['call_number','page_51']])


## 3. Seven-page union, multiplicity and ownership maps


In [ ]:
def coverage(m,root,side):
 out={}; inventory=[]
 for pg,g in m[m.page.isin(PAGES)].groupby('page'):
  g=g.sort_values('call_number').reset_index(drop=True); u=n=fo=lo=None
  for ordinal,r in enumerate(g.itertuples(index=False),1):
   p=cp(root,r.call_number,'mask.npy'); req(p.exists(),f'Missing {p}'); raw=np.load(p,allow_pickle=False,mmap_mode='r'); q=np.asarray(raw)>0
   req(bh(raw)==r.mask_hash,f'{side} {r.call_number}: hash mismatch')
   if u is None: u=np.zeros(q.shape,bool); n=np.zeros(q.shape,np.uint16); fo=np.zeros(q.shape,np.int32); lo=np.zeros(q.shape,np.int32)
   req(q.shape==u.shape,f'{side} {pg}: shape mismatch'); u|=q; n+=q.astype(np.uint16); fo[(fo==0)&q]=ordinal; lo[q]=ordinal
   inventory.append({'side':side,'page':pg,'call_number':int(r.call_number),'ordinal':ordinal,'pixels':int(q.sum())})
  out[pg]={'union':u,'n':n,'first':fo,'last':lo,'calls':g.call_number.astype(int).tolist()}; print(side,pg,len(g),int(u.sum()),int(n.sum()))
 return out,pd.DataFrame(inventory)
c51,i51=coverage(m51,CAPTURE_51,'51R2'); c53,i53=coverage(m53,CAPTURE_53,'53R2'); pd.concat([i51,i53]).to_csv(T/'seven_page_call_inventory.csv',index=False)
rows=[]; uc=[]; mc=[]; oc=[]
for pg in PAGES:
 a,b=c51[pg],c53[pg]; ux=a['union']^b['union']; nx=a['n']!=b['n']; fx=a['first']!=b['first']; lx=a['last']!=b['last']
 rows.append({'page':pg,'calls_51':len(a['calls']),'calls_53':len(b['calls']),'union_pixels_51':int(a['union'].sum()),'union_pixels_53':int(b['union'].sum()),'union_xor_pixels':int(ux.sum()),'memberships_51':int(a['n'].sum()),'memberships_53':int(b['n'].sum()),'multiplicity_difference_pixels':int(nx.sum()),'first_owner_difference_pixels':int(fx.sum()),'last_owner_difference_pixels':int(lx.sum())})
 for y,x in zip(*np.where(ux)): uc.append({'page':pg,'y':int(y),'x':int(x),'in_51':bool(a['union'][y,x]),'in_53':bool(b['union'][y,x])})
 for y,x in zip(*np.where(nx)): mc.append({'page':pg,'y':int(y),'x':int(x),'n_51':int(a['n'][y,x]),'n_53':int(b['n'][y,x])})
 for y,x in zip(*np.where(fx|lx)): oc.append({'page':pg,'y':int(y),'x':int(x),'first_51':int(a['first'][y,x]),'first_53':int(b['first'][y,x]),'last_51':int(a['last'][y,x]),'last_53':int(b['last'][y,x])})
 for name,q in [('union',ux),('multiplicity',nx),('first_owner',fx),('last_owner',lx)]: cv2.imwrite(str(IMG/f'{pg}_{name}_difference.png'),q.astype(np.uint8)*255)
coverage_summary=pd.DataFrame(rows); coverage_summary.to_csv(T/'page_call_coverage_summary.csv',index=False); pd.DataFrame(uc).to_csv(T/'union_coordinate_differences.csv',index=False); pd.DataFrame(mc).to_csv(T/'multiplicity_coordinate_differences.csv',index=False); pd.DataFrame(oc).to_csv(T/'ownership_coordinate_differences.csv',index=False); display(coverage_summary)


## 4. Classify all 50 coordinates against the complete call population


In [ ]:
req(COORDINATES.exists(),f'Missing {COORDINATES}'); xy=pd.read_csv(COORDINATES)
for c in ['call_number','y','x']: xy[c]=pd.to_numeric(xy[c],errors='raise').astype(int)
xy['page']=xy.page.map(page); req(len(xy)==50,f'Expected 50 rows, got {len(xy)}')
def owners(pg,y,x,inv,root):
 o=[]
 for r in inv[inv.page==pg].itertuples(index=False):
  a=np.load(cp(root,r.call_number,'mask.npy'),allow_pickle=False,mmap_mode='r')
  if a[y,x]>0:o.append((int(r.ordinal),int(r.call_number)))
 return o
out=[]
for r in xy.itertuples(index=False):
 a,b=c51[r.page],c53[r.page]; o51=owners(r.page,r.y,r.x,i51,CAPTURE_51); o53=owners(r.page,r.y,r.x,i53,CAPTURE_53); n51=int(a['n'][r.y,r.x]); n53=int(b['n'][r.y,r.x])
 if not a['union'][r.y,r.x] and b['union'][r.y,r.x]: d='PAGE_LEVEL_53_EXPANSION'
 elif n53>n51:d='EXTRA_53_OVERLAP'
 elif o51!=o53:d='REGIONAL_REASSIGNMENT'
 elif a['first'][r.y,r.x]!=b['first'][r.y,r.x] or a['last'][r.y,r.x]!=b['last'][r.y,r.x]:d='CALL_ORDER_CHANGE'
 else:d='UNRESOLVED'
 out.append({'call_number':r.call_number,'page':r.page,'y':r.y,'x':r.x,'multiplicity_51':n51,'multiplicity_53':n53,'owners_51':'|'.join(f'{o}:{c}' for o,c in o51),'owners_53':'|'.join(f'{o}:{c}' for o,c in o53),'diagnosis':d})
own50=pd.DataFrame(out); own50.to_csv(T/'fifty_pixel_ownership_analysis.csv',index=False); osum=own50.groupby('diagnosis',as_index=False).size().rename(columns={'size':'pixels'}); osum.to_csv(T/'fifty_pixel_ownership_summary.csv',index=False); display(osum)


## 5. Captured-stream replay and nine-mask counterfactual

Each page is rebuilt in captured order. Original replay writes the captured raw result only where that call's mask is positive. Counterfactual replay changes only the nine 53R2 calls: it uses the corresponding 51R2 mask and recalculates Telea from the recorded 53R2 source.


In [ ]:
def replay(pg,m,root,side,sub=None):
 g=m[m.page==pg].sort_values('call_number'); req(len(g)>0,f'{side} {pg}: no calls'); first=int(g.iloc[0].call_number); obs=np.load(cp(root,first,'source.npy'),allow_pickle=False); result=obs.copy()
 for r in g.itertuples(index=False):
  call=int(r.call_number); mp=cp(root,call,'mask.npy'); rp=cp(root,call,'result.npy'); req(mp.exists() and rp.exists(),f'{side} call {call}: missing capture array')
  if sub and call in sub:
   mask=np.load(sub[call],allow_pickle=False)>0; src=np.load(cp(root,call,'source.npy'),allow_pickle=False); raw=cv2.inpaint(src,mask.astype(np.uint8)*255,3,cv2.INPAINT_TELEA)
  else:
   mask=np.load(mp,allow_pickle=False,mmap_mode='r')>0; raw=np.load(rp,allow_pickle=False,mmap_mode='r')
  req(mask.shape==result.shape and raw.shape==result.shape,f'{side} call {call}: shape mismatch'); result[mask]=raw[mask]
 return obs,result
sub={int(r.call_number):cp(CAPTURE_51,int(r.call_number),'mask.npy') for r in diff.itertuples(index=False)}
rr=[]
for pg in PAGES:
 o51,r51=replay(pg,m51,CAPTURE_51,'51R2'); o53,r53=replay(pg,m53,CAPTURE_53,'53R2'); req(np.array_equal(o51,o53),f'{pg}: observed inputs differ'); _,rcf=replay(pg,m53,CAPTURE_53,'53R2 counterfactual',sub)
 d=r51!=r53; q=r51!=rcf; rr.append({'page':pg,'51_vs_53_different_pixels':int(d.sum()),'51_vs_53_max_abs_difference':int(np.abs(r51.astype(np.int16)-r53.astype(np.int16)).max()),'51_vs_counterfactual_different_pixels':int(q.sum()),'51_vs_counterfactual_max_abs_difference':int(np.abs(r51.astype(np.int16)-rcf.astype(np.int16)).max()),'counterfactual_exact':bool(np.array_equal(r51,rcf))})
 cv2.imwrite(str(IMG/f'{pg}_51_vs_53_absdifference.png'),np.abs(r51.astype(np.int16)-r53.astype(np.int16)).astype(np.uint8)); cv2.imwrite(str(IMG/f'{pg}_51_vs_counterfactual_absdifference.png'),np.abs(r51.astype(np.int16)-rcf.astype(np.int16)).astype(np.uint8)); del o51,o53,r51,r53,rcf; gc.collect()
replay_df=pd.DataFrame(rr); replay_df.to_csv(T/'counterfactual_replay_summary.csv',index=False); display(replay_df)


## 6. Final diagnosis, correction class and regression gates


In [ ]:
ux=int(coverage_summary.union_xor_pixels.sum()); nx=int(coverage_summary.multiplicity_difference_pixels.sum()); ox=int((coverage_summary.first_owner_difference_pixels+coverage_summary.last_owner_difference_pixels).sum()); exact=bool(replay_df.counterfactual_exact.all())
if ux>0:
 diagnosis='PAGE_LEVEL_REGIONAL_MASK_EXPANSION'; correction='Constrain 53R2 regional Telea masks to the page-level authoritative mask and reproduce 51R2 crop/canvas clipping. Assert regional-mask union equals the page intervention mask.'
elif nx>0:
 diagnosis='REGIONAL_OVERLAP_OR_MULTIPLICITY'; correction='Remove extra regional membership at the measured coordinates. Reproduce 51R2 half-open slice and crop-placement semantics so each page pixel has the same call multiplicity and ownership.'
elif ox>0:
 diagnosis='REGIONAL_OWNER_OR_ORDER'; correction='Keep page union and multiplicity unchanged but reproduce 51R2 regional ownership and page-local call order.'
else:
 diagnosis='COMPOSITION_SOURCE_STATE'; correction='Mask coverage and ownership are exact. Reproduce whether 51R2 calls read the immutable observed page or the evolving composed page.'
correction += (' Nine-mask substitution reproduces every 51R2 page, so this is sufficient at the captured-call boundary.' if exact else ' Nine-mask substitution does not reproduce every 51R2 page. Inspect remaining counterfactual residuals before editing 53R2.')
verdict={'created_utc':datetime.now(timezone.utc).isoformat(),'status':'READY_TO_PATCH_53R2' if exact else 'MORE_DIAGNOSIS_REQUIRED','diagnosis':diagnosis,'recommended_correction':correction,'page_union_xor_pixels':ux,'multiplicity_difference_pixels':nx,'ownership_difference_count':ox,'fifty_pixel_classes':own50.diagnosis.value_counts().to_dict(),'counterfactual_exact_all_seven_pages':exact}
(OUT/'stage_f_final_forensic_verdict.json').write_text(json.dumps(verdict,indent=2)); print(json.dumps(verdict,indent=2))
reg={'calls_each_side':2718,'allowed_source_hash_mismatches':0,'allowed_mask_hash_mismatches':0,'allowed_radius_mismatches':0,'allowed_flags_mismatches':0,'allowed_raw_result_hash_mismatches':0,'allowed_corrected_page_different_pixels':0,'allowed_outside_mask_drift':0,'rerun_notebook_54_unchanged':True}; (OUT/'corrected_53R2_regression_spec.json').write_text(json.dumps(reg,indent=2))


## 7. Final Upload Package


In [ ]:
required=[T/'nine_differing_calls.csv',T/'page_call_coverage_summary.csv',T/'fifty_pixel_ownership_analysis.csv',T/'fifty_pixel_ownership_summary.csv',T/'counterfactual_replay_summary.csv',OUT/'stage_f_final_forensic_verdict.json',OUT/'corrected_53R2_regression_spec.json']
for p in required:req(p.exists(),f'Missing {p}')
for p in required:
 d=UP/('tables' if p.suffix=='.csv' else 'audit'); d.mkdir(parents=True,exist_ok=True); shutil.copy2(p,d/p.name)
(UP/'images').mkdir(exist_ok=True)
for p in IMG.glob('*difference*.png'):shutil.copy2(p,UP/'images'/p.name)
(UP/'README.txt').write_text('Start with audit/stage_f_final_forensic_verdict.json and tables/counterfactual_replay_summary.csv. Capture arrays remain in their original directories.')
inv=[]
for p in sorted(UP.rglob('*')):
 if p.is_file():inv.append({'relative_path':str(p.relative_to(UP)),'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()})
pd.DataFrame(inv).to_csv(UP/'upload_manifest.csv',index=False); z=shutil.make_archive(str(OUT/'stage_f_final_upload_bundle'),'zip',UP); print('Run',OUT); print('Upload bundle',z)
